In [1]:
import pandas as pd

df = pd.read_csv("../raw/Orientamento_sessuale.csv")

df.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,SEX,Sesso,SEXUAL_ORIENTATION,Orientamento sessuale,TIME_PERIOD,Osservazione,OBS_STATUS,Stato_osservazione
0,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,OMOX,Omosessuale,2015.0,24.0,NaN,NaN
1,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,OMOX,Omosessuale,2016.0,34.0,NaN,NaN
2,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,OMOX,Omosessuale,2017.0,56.0,NaN,NaN
3,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,OMOX,Omosessuale,2018.0,36.0,NaN,NaN
4,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,OMOX,Omosessuale,2019.0,45.0,NaN,NaN


In [2]:
# Per ogni colonna: numero di valori unici + elenco valori

for col in df.columns:
    print("=" * 60)
    print(f"Colonna: {col}")
    print(f"Numero valori unici: {df[col].nunique(dropna=False)}")
    print("Valori unici:")
    print(df[col].unique())
    print("\n")

Colonna: FREQ
Numero valori unici: 61
Valori unici:
['A'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,MISSING,Missing,2023,10,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,MISSING,Missing,2024,6,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Altro o non indicato,2015,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Altro o non indicato,2016,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Altro o non indicato,2017,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Altro o non indicato,2018,3,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Altro o non indicato,2019,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,OTHER,Alt

In [3]:
# Rimozione colonne non informative o ridondanti:
# - colonne costanti (un solo valore per tutto il dataset)
# - colonne duplicate sul piano informativo
# - colonne completamente vuote

cols_to_drop = [
    "FREQ",               
    "Frequenza",          
    "REF_AREA",          
    "DATA_TYPE",                  
    "SEX",                
    "SEXUAL_ORIENTATION",     
    "OBS_STATUS",         
    "Stato_osservazione" 
]

df = df.drop(columns=cols_to_drop)

# Verifica struttura finale
print("Nuova shape:", df.shape)
print("Colonne rimanenti:", df.columns.tolist())


Nuova shape: (2640, 6)
Colonne rimanenti: ['Territorio', 'Indicatore', 'Sesso', 'Orientamento sessuale', 'TIME_PERIOD', 'Osservazione']


In [4]:
# Standardizzazione nomi colonne:
# - tutto minuscolo
# - nomi più chiari e coerenti

df = df.rename(columns={
    "Territorio": "territorio",
    "Indicatore": "canale_segnalazione",
    "Sesso": "sesso",
    "Orientamento sessuale": "orientamento_sessuale",
    "TIME_PERIOD": "periodo",
    "Osservazione": "numero_vittime"
})


# Eliminazione delle righe completamente vuote 
df = df.dropna(how="all")

# Stampa delle colonne e dei valori unici che contengono
for col in df.columns:
    print("\nColonna:", col)
    print(df[col].unique())


Colonna: territorio
['Italia' 'Nord-ovest' 'Piemonte' 'Liguria' 'Lombardia' 'Nord-est'
 'Trentino Alto Adige / Südtirol' 'Veneto' 'Friuli-Venezia Giulia'
 'Emilia-Romagna' 'Centro' 'Toscana' 'Umbria' 'Marche' 'Lazio' 'Sud'
 'Abruzzo' 'Molise' 'Campania' 'Puglia' 'Basilicata' 'Calabria' 'Isole'
 'Sicilia' 'Sardegna' 'Non indicato']

Colonna: canale_segnalazione
['Utenti del 1522']

Colonna: sesso
['Maschi' 'Femmine' 'Non indicato' 'Totale']

Colonna: orientamento_sessuale
['Omosessuale' 'Transessuale' 'Missing' 'Altro o non indicato' 'Totale']

Colonna: periodo
[2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022. 2023. 2024.]

Colonna: numero_vittime
[   24.    34.    56. ...  6921. 10274.  8771.]


In [5]:
# Pulizia colonna territorio:
# - uniformazione minuscolo
# - correzione denominazioni incoerenti / verbose

df["territorio"] = (
    df["territorio"]
    .str.lower()
    .str.strip()
    .str.replace("-", " ", regex=False)
)

df["territorio"] = df["territorio"].replace({
    "trentino alto adige / südtirol": "trentino alto adige"
})


# Uniformazione minuscolo per la colonna sesso e orientamento_sessuale
df["sesso"] = df["sesso"].str.lower().str.strip()
df["orientamento_sessuale"] = df["orientamento_sessuale"].str.lower().str.strip()


# Rimozione della voce aggregata "totale" 
# da sesso e orientamento_sessuale per evitare doppio conteggio

df = df[
    (df["sesso"] != "totale") &
    (df["orientamento_sessuale"] != "totale")
]


# Conversione delle colonne numeriche da float a integer

df["periodo"] = df["periodo"].astype(int)
df["numero_vittime"] = df["numero_vittime"].astype(int)


# Verifica finale post-pulizia (struttura + qualità base)

# 1) Dimensioni e info colonne
display(df.head(10))
df.info()

# 2) Tipi e missing
print("\nMissing values per colonna:")
display(df.isna().sum().sort_values(ascending=False))

# 3) Valori unici (conteggio) per colonna
print("\nNumero valori unici per colonna:")
display(df.nunique(dropna=False).sort_values(ascending=False))

# 4) Check duplicati (righe identiche)
print("\nRighe duplicate (identiche su tutte le colonne):", df.duplicated().sum())

# 5) Valori distinti per ogni colonna
for col in df.columns:
    print("\nColonna:", col)
    
    if col == "periodo":
        print(sorted(df[col].astype(int).unique().tolist()))
    else:
        print(df[col].unique())

,territorio,canale_segnalazione,sesso,orientamento_sessuale,periodo,numero_vittime
0,italia,Utenti del 1522,maschi,omosessuale,2015,24
1,italia,Utenti del 1522,maschi,omosessuale,2016,34
2,italia,Utenti del 1522,maschi,omosessuale,2017,56
3,italia,Utenti del 1522,maschi,omosessuale,2018,36
4,italia,Utenti del 1522,maschi,omosessuale,2019,45
5,italia,Utenti del 1522,maschi,omosessuale,2020,91
6,italia,Utenti del 1522,maschi,omosessuale,2021,92
7,italia,Utenti del 1522,maschi,omosessuale,2022,77
8,italia,Utenti del 1522,maschi,transessuale,2015,14
9,italia,Utenti del 1522,maschi,transessuale,2016,12


<class 'pandas.core.frame.DataFrame'>
Index: 1144 entries, 0 to 2596
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   territorio             1144 non-null   object
 1   canale_segnalazione    1144 non-null   object
 2   sesso                  1144 non-null   object
 3   orientamento_sessuale  1144 non-null   object
 4   periodo                1144 non-null   int64 
 5   numero_vittime         1144 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 62.6+ KB

Missing values per colonna:


territorio               0
canale_segnalazione      0
sesso                    0
orientamento_sessuale    0
periodo                  0
numero_vittime           0
dtype: int64


Numero valori unici per colonna:


numero_vittime           442
territorio                26
periodo                   10
orientamento_sessuale      4
sesso                      3
canale_segnalazione        1
dtype: int64


Righe duplicate (identiche su tutte le colonne): 0

Colonna: territorio
['italia' 'nord ovest' 'piemonte' 'liguria' 'lombardia' 'nord est'
 'trentino alto adige' 'veneto' 'friuli venezia giulia' 'emilia romagna'
 'centro' 'toscana' 'umbria' 'marche' 'lazio' 'sud' 'abruzzo' 'molise'
 'campania' 'puglia' 'basilicata' 'calabria' 'isole' 'sicilia' 'sardegna'
 'non indicato']

Colonna: canale_segnalazione
['Utenti del 1522']

Colonna: sesso
['maschi' 'femmine' 'non indicato']

Colonna: orientamento_sessuale
['omosessuale' 'transessuale' 'missing' 'altro o non indicato']

Colonna: periodo
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Colonna: numero_vittime
[   24    34    56    36    45    91    92    77    14    12    32    15
    60    31    25  6154  8278  2166  1648  2057  2516  2290  3275  3901
  3755    13    20    16    28    89    81    44    40   183   125    70
 41235 52166 17452 15887 15441 20629 18903 28021 31821 28445  4324  4604
     1     2     9    11    10 

In [6]:
# Salvataggio dataset pulito

output_path = "../data_clean/Orientamento_sessuale.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("File salvato in:", output_path)

File salvato in: ../data_clean/Orientamento_sessuale.csv
